# RAG_DB: build the CraveCart vector database

Turns the 18 Markdown documents in `knowledge-base/` into a Chroma vector database, like the class `RAG_DB.ipynb`, with three upgrades:

1. **Heading-aware chunks:** documents are split at `#`, `##` and `###` headings first, so a chunk never mixes two policies, and every chunk is labelled with its document and section. The app uses those labels for citations.
2. **Better embeddings:** `BAAI/bge-small-en-v1.5` (same size as `all-MiniLM-L6-v2`, better at retrieval).
3. **Self-refreshing:** a fingerprint of the files is saved with the database, so it rebuilds only when a document changes.

Run the cells in order. In Colab, the first cell clones the project and installs everything.

## Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SAVE_DB_TO_DRIVE = False  # True: keep the vector DB in Google Drive, like the class notebook

if IN_COLAB:
    if not Path("ai-engineering-projects").exists():
        !git clone --depth 1 https://github.com/C0derANANT/ai-engineering-projects.git
    %cd ai-engineering-projects/03-Food-Delivery-RAG
    %pip install -q -r requirements.txt
    if SAVE_DB_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["VECTOR_DB_DIR"] = "/content/drive/MyDrive/ai/cravecart_vector_db"
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # run from the project folder so `rag` can be imported

sys.path.insert(0, str(Path.cwd()))
print("Project folder:", Path.cwd())

## 1. Load the documents
Each subfolder of `knowledge-base/` becomes the `doc_type` metadata, as in the class loader.

In [ ]:
from collections import Counter
from rag.config import Settings
from rag.knowledge_base import load_documents

settings = Settings()
documents = load_documents(settings.knowledge_base_dir)
print(f"Loaded {len(documents)} documents")
for doc_type, count in sorted(Counter(d.metadata["doc_type"] for d in documents).items()):
    print(f"  {doc_type:15s} {count}")

## 2. Split into heading-aware chunks

In [ ]:
from rag.knowledge_base import split_documents

chunks = split_documents(documents, settings.chunk_size, settings.chunk_overlap)
sizes = [len(c.page_content) for c in chunks]
print(f"Divided into {len(chunks)} chunks (average {sum(sizes) // len(sizes)} characters)")

example = next(c for c in chunks if c.metadata["source"] == "policies/refund-policy.md" and "5.1" in c.metadata["section"])
print("\nExample chunk metadata:", {k: example.metadata[k] for k in ("doc_type", "source", "section")})
print(example.page_content[:500])

## 3. Create the embeddings and the Chroma database
The first run downloads the embedding model (about 130 MB) and takes about a minute.

In [ ]:
from rag.knowledge_base import open_vector_store

vectorstore, info = open_vector_store(settings, force_rebuild=True)
print(f"Vector store created with {info.chunks} chunks from {info.files} files in {info.seconds:.0f}s")
print("Saved at:", settings.vector_db_dir)

## 4. Test retrieval
Each result shows its relevance score and the document › section it came from.

In [ ]:
from rag.knowledge_base import search

questions = [
    "How much is the cancellation fee after the restaurant accepts my order?",
    "What does error CC-PAY-302 mean?",
    "How many earned leave days can I carry forward?",
]
for q in questions:
    print(f"\nQ: {q}")
    for hit in search(vectorstore, q, k=3):
        print(f"  [{hit.number}] {hit.score:.2f}  {hit.label}")

Next: open **RAG_APP.ipynb** to chat with the knowledge base.